# Projet Leyenda - Livrable 2
##### *Par Noah.V, Noah.M, Hugo.L*

## **Contexte:**
L'entreprise TouNum est une entreprise qui travaille sur la numérisation de différents type de documents. Ils essayent de moderniser leur architecture pour le tri automatique de documents due au grand nombre de documents fourni par les clients. Faisant appel à nous pour determiner le captionning des images avec légendes de manière automatiques. Nous remarquons deux contraintes, la première est la qualité des images qui diffère par exemple entre chaque image, et trié automatiquement si c'est une photo ou d'autres types de documents.

Le workflow demandé par l'entreprise devra suivre ce modèle la:

![Image demandé](src/workflow.png)

Le projet sera divisé en trois partie:
- La première sera de contrsuire un CNN via des datasets d'images pré étiqueté
- Le deuxième sera d'améliorer la clarté et l'enlever les différents bruits ou flou des images
- Le troisème sera de créer un réseau de neurones qui génère des légendes pour des photographies

Nous avons déjà réalisé le premier livrable avec différentes architectures, mais aussi différentes résolutions et comme on a pu le voir dans ce précédent livrable, les images redimensionné peuvent avoir du bruit ou être floue. C'est pour cela que dans ce ***deuxième livrable***, le but sera d'améliorer la clarté de ses images et permettre au modèle de mieux comprendre les images pour une meilleure detection.

Détail du ***Livrable 2:***
- Le code de chargement du fichier
- Le code du débruitage sur un sous-ensemble d'images bruitées. Le code doit être accompagné d'explications.
- Une étude de cas.

## Importation des librairies

In [35]:
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences


## Etape 1 : Chargement des données



In [ ]:
DATA_DIR = Path("Data")

ANNOTATION_FILES = {
    "train": DATA_DIR / "annotations" / "captions_train2014.json",
    "validation": DATA_DIR / "annotations" / "captions_val2014.json",
}

IMAGE_DIRS = {
    "train": DATA_DIR / "train2014",
    "validation": DATA_DIR / "val2014",
}


def load_split(split):
    with ANNOTATION_FILES[split].open("r", encoding="utf-8") as f:
        data = json.load(f)

    
    records = {}
    for image in data["images"]:
        image_id = image["id"]
        if image_id in records:
            raise ValueError(f"{split} : ID image dupliqué {image_id}")
        records[image_id] = {
            "image_id": image_id,
            "image_path": IMAGE_DIRS[split] / image["file_name"],
            "captions": [],
        }

    
    for ann in data["annotations"]:
        image_id = ann["image_id"]
        if image_id not in records:
            raise ValueError(f"{split} : légende liée à une image inconnue {image_id}")
        records[image_id]["captions"].append(ann["caption"])

    
    missing_files = [r["image_path"] for r in records.values() if not r["image_path"].is_file()]
    no_caption = [i for i, r in records.items() if not r["captions"]]
    n_captions = sum(len(r["captions"]) for r in records.values())

    print(f"{split} : {len(records):,} images, {n_captions:,} légendes")
    print(f"  images introuvables sur disque : {len(missing_files)}")
    print(f"  images sans légende            : {len(no_caption)}")

    if missing_files:
        print("  exemples manquants :", missing_files[:3])

    return records


train_records = load_split("train")
val_records = load_split("validation")

shared = set(train_records) & set(val_records)
print("IDs communs train/validation :", len(shared))

train : 82,783 images, 414,113 légendes
  images introuvables sur disque : 0
  images sans légende            : 0
validation : 40,504 images, 202,654 légendes
  images introuvables sur disque : 0
  images sans légende            : 0
IDs communs train/validation : 0


In [9]:
example = next(iter(train_records.values()))

print("ID image :", example["image_id"])
print("Chemin :", example["image_path"])

for caption in example["captions"]:
    print("-", caption)

ID image : 57870
Chemin : Data\train2014\COCO_train2014_000000057870.jpg
- A restaurant has modern wooden tables and chairs.
- A long restaurant table with rattan rounded back chairs.
- a long table with a plant on top of it surrounded with wooden chairs 
- A long table with a flower arrangement in the middle for meetings
- A table is adorned with wooden chairs with blue accents.


# Etape 2 : Prétraitement des données

In [21]:
from tensorflow.keras.applications.inception_v3 import (
    InceptionV3,
    preprocess_input,
)

In [ ]:
IMAGE_SIZE = (299, 299)
BATCH_SIZE = 32

inception = InceptionV3(
    weights="imagenet",
    include_top=False,
    pooling=None,
    input_shape=(*IMAGE_SIZE, 3),
)

inception.trainable = False

inputs = Input(
    shape=(*IMAGE_SIZE, 3),
    name="image_input",
)


outputs = inception(inputs, training=False)

image_encoder = Model(
    inputs=inputs,
    outputs=outputs,
    name="image_encoder",
)

print("Entrée :", image_encoder.input_shape)
print("Sortie :", image_encoder.output_shape)

Entrée : (None, 299, 299, 3)
Sortie : (None, 8, 8, 2048)


Charger et preprocess 

In [23]:
def load_and_preprocess_image(image_id, image_path):
    image_bytes = tf.io.read_file(image_path)

    image = tf.io.decode_jpeg(
        image_bytes,
        channels=3,
    )

    image = tf.image.resize(image, IMAGE_SIZE)
    image = preprocess_input(image)

    return image_id, image

Créer le dataset avec le lien image id et image path

In [24]:
def make_image_dataset(records):
    image_ids = np.array(list(records.keys()), dtype=np.int64)

    image_paths = [
        str(records[int(image_id)]["image_path"])
        for image_id in image_ids
    ]

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_ids, image_paths)
    )

    dataset = dataset.map(
        load_and_preprocess_image,
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


train_image_dataset = make_image_dataset(train_records)
val_image_dataset = make_image_dataset(val_records)

print("Datasets train et validation créés.")

Datasets train et validation créés.


Test avec un batch

In [ ]:

batch_ids, batch_images = next(iter(train_image_dataset))

batch_features = image_encoder(batch_images, training=False)

batch_features = tf.reshape(
    batch_features,
    (tf.shape(batch_features)[0], 64, 2048),
)

print("Caractéristiques :", batch_features.shape)
# Attendu : (32, 64, 2048) si BATCH_SIZE = 32

Caractéristiques : (32, 64, 2048)


## Extraction des features

In [ ]:
FEATURES_DIR = Path("features/inception_grid_fp16")


def extract_features(records, split_name, limit=None):
    out_dir = FEATURES_DIR / split_name
    out_dir.mkdir(parents=True, exist_ok=True)

    pending = [i for i in records if not (out_dir / f"{i}.npy").exists()]
    already_done = len(records) - len(pending)
    if limit is not None:
        pending = pending[:limit]

    print(f"{split_name} : {already_done:,} déjà faites, {len(pending):,} à traiter")
    if not pending:
        return

    dataset = make_image_dataset({i: records[i] for i in pending})
    total_batches = (len(pending) + BATCH_SIZE - 1) // BATCH_SIZE

    start = time.time()
    done = 0

    for batch_number, (image_ids, images) in enumerate(dataset, start=1):
        features = image_encoder(images, training=False)                      
        features = tf.reshape(features, (tf.shape(features)[0], 64, 2048))    
        features = features.numpy().astype(np.float16)

        for image_id, vector in zip(image_ids.numpy(), features):
            tmp_path = out_dir / f"{image_id}.tmp.npy"
            np.save(tmp_path, vector)
            tmp_path.replace(out_dir / f"{image_id}.npy")   

        done += len(features)

        if batch_number % 100 == 0 or batch_number == total_batches:
            speed = done / (time.time() - start)
            print(f"{split_name} : {batch_number}/{total_batches} batches, {speed:.1f} images/s", flush=True)

In [ ]:

extract_features(train_records, "train")
extract_features(val_records, "validation")

In [ ]:
for split, records in [("train", train_records), ("validation", val_records)]:
    out_dir = FEATURES_DIR / split
    files = [p for p in out_dir.glob("*.npy") if not p.name.endswith(".tmp.npy")]
    print(f"{split} : {len(files):,} fichiers pour {len(records):,} images")

example = np.load(files[0])
print("Forme :", example.shape, "| type :", example.dtype)   # attendu : (64, 2048) float16

Prépare caption

In [26]:
def prepare_captions(records):
    image_ids = []
    captions = []

    for image_id, record in records.items():
        for caption in record["captions"]:
            image_ids.append(image_id)
            captions.append(f"startseq {caption.strip()} endseq")

    return np.asarray(image_ids, dtype=np.int64), captions


train_caption_image_ids, train_captions = prepare_captions(train_records)
val_caption_image_ids, val_captions = prepare_captions(val_records)

print("Légendes train :", len(train_captions))
print("Légendes validation :", len(val_captions))

Légendes train : 414113
Légendes validation : 202654


Tokenizer

In [ ]:
TOP_K = 5000

tokenizer = Tokenizer(num_words=TOP_K, oov_token="unk")
tokenizer.fit_on_texts(train_captions)   

train_sequences = tokenizer.texts_to_sequences(train_captions)
val_sequences = tokenizer.texts_to_sequences(val_captions)

vocab_size = min(TOP_K, len(tokenizer.word_index))

start_token_id = tokenizer.word_index["startseq"]
end_token_id = tokenizer.word_index["endseq"]

assert start_token_id < vocab_size
assert end_token_id < vocab_size

print("vocab_size :", vocab_size)
print("startseq :", start_token_id, "| endseq :", end_token_id)

vocab_size : 5000
startseq : 3 | endseq : 4


In [28]:
max_length = max(len(sequence) for sequence in train_sequences)

val_sequences = [
    sequence
    if len(sequence) <= max_length
    else sequence[:max_length - 1] + [end_token_id]
    for sequence in val_sequences
]

train_caption_tokens = pad_sequences(
    train_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post",
    value=0,
    dtype="int32",
)

val_caption_tokens = pad_sequences(
    val_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post",
    value=0,
    dtype="int32",
)

print("Longueur maximale :", max_length)
print("Train :", train_caption_tokens.shape)
print("Validation :", val_caption_tokens.shape)

Longueur maximale : 52
Train : (414113, 52)
Validation : (202654, 52)


In [30]:
print("vocab_size :", vocab_size)
print("max_length :", max_length)
print("indice max dans train :", train_caption_tokens.max())
print("indice max dans validation :", val_caption_tokens.max())
print("unk :", tokenizer.word_index["unk"],
      "| startseq :", start_token_id,
      "| endseq :", end_token_id)

assert train_caption_tokens.max() < vocab_size
assert val_caption_tokens.max() < vocab_size

vocab_size : 5000
max_length : 52
indice max dans train : 4999
indice max dans validation : 4999
unk : 1 | startseq : 3 | endseq : 4


Encodeur CNN

In [ ]:
EMBEDDING_DIM = 256   


class CNN_Encoder(tf.keras.Model):
    def __init__(self, embedding_dim):
        super().__init__()
        self.fc = tf.keras.layers.Dense(embedding_dim)

    def call(self, x):
        x = self.fc(x)        
        x = tf.nn.relu(x)
        return x


encoder = CNN_Encoder(EMBEDDING_DIM)

## Mécanisme d’attention de Bahdanau

In [ ]:
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()

        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, features, hidden):
        )

        hidden_with_time_axis = tf.expand_dims(hidden, axis=1)

        
        attention_hidden = tf.nn.tanh(
            self.W1(features) + self.W2(hidden_with_time_axis)
        )

        
        scores = self.V(attention_hidden)

        
        attention_weights = tf.nn.softmax(scores, axis=1)

        
        context_vector = tf.reduce_sum(
            attention_weights * features,
            axis=1,
        )

        return context_vector, attention_weights

In [ ]:

RNN_UNITS = 512
ATTENTION_UNITS = 512


attention_test = BahdanauAttention(ATTENTION_UNITS)

encoded_features = encoder(batch_features)


hidden_test = tf.zeros(
    (tf.shape(encoded_features)[0], RNN_UNITS),
    dtype=encoded_features.dtype,
)

context_vector, attention_weights = attention_test(
    encoded_features,
    hidden_test,
)

print("Caractéristiques encodées :", encoded_features.shape)
print("État caché :", hidden_test.shape)
print("Vecteur de contexte :", context_vector.shape)
print("Poids d’attention :", attention_weights.shape)

tf.debugging.assert_near(
    tf.reduce_sum(attention_weights, axis=1),
    tf.ones_like(attention_weights[:, 0, :]),
    atol=1e-5,
)

tf.debugging.assert_all_finite(
    context_vector,
    "Le vecteur de contexte contient des valeurs invalides.",
)

Caractéristiques encodées : (32, 64, 256)
État caché : (32, 512)
Vecteur de contexte : (32, 256)
Poids d’attention : (32, 64, 1)


<tf.Tensor: shape=(32, 256), dtype=float32, numpy=
array([[0.90259266, 0.04848988, 0.01264063, ..., 0.12390804, 0.489711  ,
        0.33525878],
       [1.3874277 , 0.05899088, 0.3068332 , ..., 0.88598645, 0.7031134 ,
        0.8477063 ],
       [1.0394372 , 0.19451872, 0.06785088, ..., 0.0236203 , 0.41495532,
        0.21100898],
       ...,
       [1.6262337 , 0.05455462, 0.06073774, ..., 0.9171927 , 0.581483  ,
        0.30991763],
       [1.7219821 , 0.05985732, 0.20027377, ..., 0.3324566 , 0.55333704,
        0.8985964 ],
       [1.907786  , 0.07078156, 0.07874645, ..., 1.3862106 , 0.0506932 ,
        1.0699008 ]], dtype=float32)>